In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate a synthetic binary classification task with noise
X, y = make_classification(
    n_samples=60_000,
    n_features=25,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    flip_y=0.03,
    random_state=42,
)

# 3-Way Split: Train (60%), Validation (20%), Test (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

gbc = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    validation_fraction=0.20,  # Uses internal validation split from train
    n_iter_no_change=15,        # Early stopping parameter in scikit-learn
    tol=1e-4,
    random_state=42,
)

gbc.fit(X_train, y_train)

auc_gbc = roc_auc_score(y_test, gbc.predict_proba(X_test)[:, 1])
print(f"GBC Stopped at Tree: {gbc.n_estimators_} | Test AUC: {auc_gbc:.4f}")

GBC Stopped at Tree: 637 | Test AUC: 0.9829


In [ ]:
# XGBoost: XGBClassifier Implementation
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=5,
    reg_lambda=1.5,           # L2 leaf weight regularization parameter
    gamma=0.1,                # Minimum split loss reduction threshold
    early_stopping_rounds=25, # Native early stopping criterion
    eval_metric="logloss",
    tree_method="hist",       # Fast histogram approximation
    random_state=42,
)

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

auc_xgb = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f"XGB Best Iteration: {xgb.best_iteration} | Test AUC: {auc_xgb:.4f}")

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgb = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,    # Primary capacity control (Leaf-wise)
    max_depth=6,      # Guardrail against deep leaf-wise overfitting
    subsample=0.8,
    random_state=42,
)

lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    callbacks=[
        early_stopping(stopping_rounds=25, verbose=False),
        log_evaluation(period=0),
    ],
)

auc_lgb = roc_auc_score(y_test, lgb.predict_proba(X_test)[:, 1])
print(f"LGBM Best Iteration: {lgb.best_iteration_} | Test AUC: {auc_lgb:.4f}")